# RUNG 10b — does a good surface AND-NOT blocker even EXIST? (CPU, no GPU)

Stop speculating, ask the atlas. arm (a)'s NOT-markers were weak (tumour-expressed). The BEST possible surface
blocker is **tumour-ABSENT but vital-HIGH**. This fetches the vital-normal expression of every tumour-absent
surface gene (~4300) and finds, **per vital tissue, whether any such blocker exists.**

- **No blocker for a vital tissue → a surface-only gate is IMPOSSIBLE there → the NOT-slot MUST be genetic** (decisive negative).
- **Blockers found → the exact candidates to put through the GPU sweep next** (possible groundbreaking surface gate).

**CPU runtime** (no GPU — saves your 4h). Resumable Drive tiles + heartbeat. Reuses your RUNG-5 tumour cache (tumour coverage of all 5007 genes, free) — same account.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists(): !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else: !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO); !git log --oneline -1; print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Drive (vital tiles + reuse RUNG-5 tumour cache) + run log
import os, os.path as p
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd = '/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG10B_CACHE'] = f'{cd}/rung10b_tiles'; os.makedirs(os.environ['RUNG10B_CACHE'], exist_ok=True)
    os.environ['LOGICGATE_CACHE'] = f'{cd}/rung5_cache.npz'   # holds tumour coverage of all 5007 surface genes
    has = p.exists(f'{cd}/rung5_cache.r5.tumour.npz')
    print('[CELL 2] Drive mounted. RUNG-5 tumour cache:', 'FOUND' if has else 'NOT FOUND -> use the account you ran RUNG-5 on')
except Exception as e:
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — tiles/cache in /content (lost on disconnect)')
    os.environ.setdefault('LOGICGATE_CACHE','/content/cancer-recon-apoptosis/data/logicgate/rung5_cache.npz')
from scripts.runlog import new_log
RUNLOG = new_log('rung10b_blocker', repo_dir='.')

In [ ]:
#@title Cell 3 — VALIDATE the screen (CPU, synthetic, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/32_surface_blocker_discovery.py','selftest'], RUNLOG)
print('[CELL 3]', '✓ validated' if rc==0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install CELLxGENE Census
import sys, importlib.util
from scripts.runlog import run_logged
for pk,pn in [('cellxgene_census','cellxgene-census'),('scanpy','scanpy')]:
    if importlib.util.find_spec(pk) is None:
        run_logged([sys.executable,'-m','pip','install','-q',pn], RUNLOG, label=f'pip {pn}')
print('[CELL 4] ✓ (restart if asked, then Run all — tiles make resume instant)')

In [ ]:
#@title Cell 5 — REAL run: fetch tumour-absent surface genes' vital expression + screen (RESUMABLE)
#@markdown CPU, ~4300 genes x vital cells. [heartbeat] every ~20s; [N/9] tissue progress. Disconnect-safe:
#@markdown re-run this cell, Drive tiles are skipped. Watch the DECISIVE line at the end.
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/32_surface_blocker_discovery.py','run'], RUNLOG)
print('[CELL 5]', '✓ done' if rc==0 else f'✗ exit {rc} — re-run to resume from last cached tissue')
from IPython.display import Image, display
if os.path.exists('runs/rung10b_blocker/rung10b_blocker.png'): display(Image('runs/rung10b_blocker/rung10b_blocker.png'))
if os.path.exists('runs/rung10b_blocker/rung10b_blocker.json'):
    d=json.load(open('runs/rung10b_blocker/rung10b_blocker.json')); print('DECISIVE:', d['DECISIVE'])

In [ ]:
#@title Cell 6 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung10b_blocker_','').replace('.log','')
b=f'/content/rung10b_run_{stem}.zip'
ps=sorted(set(glob.glob('runs/rung10b_blocker/*.png')+glob.glob('runs/rung10b_blocker/*.json')+[str(RUNLOG)]))
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')